Data Collection

restaurants & cafes in Auckland
tourist attractions in Auckland
parks in Auckland
shopping malls in Auckland

In [3]:
pip install requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


Config (env, knobs, folders)

In [5]:
# === CONFIG ===
import os, time, json, math, requests, hashlib
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Load .env (DO NOT commit .env)
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")  # <— standard name
assert API_KEY, "Set GOOGLE_API_KEY in your .env"

# City/grid settings (keep these in one place)
CITY = "Auckland"

# Crawl categories (cafés merged into restaurant downstream)
PLACE_TYPES = ["restaurant", "tourist attraction", "park", "shopping mall"]
# (Optional coverage) to also fetch 'cafe', uncomment:
# PLACE_TYPES.insert(1, "cafe")

# Rate limits 
NEARBY_NEXT_TOKEN_SLEEP = 2.0
DETAILS_SLEEP = 0.15

# Run flags (protect quota)
RUN_TEXTSEARCH = False   # set True when you actually want to fetch
RUN_DETAILS    = False   # set True when enriching details/reviews

# Data folders 
PROC  = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)
FINAL = Path("data/final");     FINAL.mkdir(parents=True, exist_ok=True)


Resilient HTTP Session (shared)

In [7]:
# One resilient session for all Google calls
SESSION = requests.Session()
retries = Retry(
    total=7, connect=7, read=7,
    backoff_factor=1.3,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET","POST"},
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=30, pool_maxsize=30)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)


Category mapper (final 4 buckets)

In [9]:
def map_to_category(google_types_str: str | None) -> str | None:
    """Collapse Google's many types to: restaurant, tourist attraction, park, shopping mall."""
    if google_types_str is None or (isinstance(google_types_str, float) and str(google_types_str) == "nan"):
        return None
    t = {x.strip().lower() for x in str(google_types_str).split(",")}
    if "shopping_mall" in t or "shopping mall" in t: return "shopping mall"
    if "tourist_attraction" in t or "tourist attraction" in t: return "tourist attraction"
    if "park" in t: return "park"
    if {"restaurant","cafe","coffee_shop","café","fast_food","meal_takeaway","meal_delivery"} & t: return "restaurant"
    return None


Text Search helpers(deduped, clean)

In [11]:
BASE_TEXT_SEARCH = "https://maps.googleapis.com/maps/api/place/textsearch/json"

def text_search(query, pagetoken=None):
    params = {"query": query, "key": API_KEY}
    if pagetoken: params["pagetoken"] = pagetoken
    r = SESSION.get(BASE_TEXT_SEARCH, params=params, timeout=(10, 30))
    r.raise_for_status()
    return r.json()

def collect_places_for_query(query, max_pages=3, sleep_between=NEARBY_NEXT_TOKEN_SLEEP):
    """Up to 3 pages. Returns minimal fields; 'google_types' not 'types'."""
    places = []
    token = None
    for _ in range(max_pages):
        if token:
            time.sleep(sleep_between)  # required delay for next_page_token
        data = text_search(query, pagetoken=token)
        results = data.get("results", [])
        for p in results:
            loc = (p.get("geometry", {}) or {}).get("location", {}) or {}
            places.append({
                "place_id": p.get("place_id"),
                "name": p.get("name"),
                "lat": loc.get("lat"),
                "lng": loc.get("lng"),
                "address": p.get("formatted_address"),
                "rating": p.get("rating"),
                "user_ratings_total": p.get("user_ratings_total"),
                "business_status": p.get("business_status"),
                "google_types": ",".join(p.get("types", [])),
                "source_category": query.split(" in ")[0].strip().lower(),  # store category only
            })
        token = data.get("next_page_token")
        if not token:
            break
    return places


Run Text Search (guarded)

In [13]:
if RUN_TEXTSEARCH:
    all_places = []
    for t in PLACE_TYPES:
        q = f"{t} in {CITY}"
        print("Fetching:", q)
        chunk = collect_places_for_query(q, max_pages=3)
        all_places.extend(chunk)
        print(f"  +{len(chunk)} (total={len(all_places)})")

    df_places = (pd.DataFrame(all_places)
                 .dropna(subset=["place_id"])
                 .drop_duplicates(subset=["place_id"])
                 .reset_index(drop=True))
    print("Unique places:", len(df_places))
    df_places.to_csv(PROC/"auckland_places_basic.csv", index=False)
else:
    basic_path = PROC/"auckland_places_basic.csv"
    assert basic_path.exists(), "Run text search once to create auckland_places_basic.csv"
    df_places = pd.read_csv(basic_path)
    print("Loaded basic places from disk:", len(df_places))


Loaded basic places from disk: 8130


Place Details (robust) + enrichment

In [21]:
BASE_PLACE_DETAILS = "https://maps.googleapis.com/maps/api/place/details/json"
DETAIL_FIELDS = (
    "name,place_id,geometry/location,formatted_address,types,"
    "rating,user_ratings_total,international_phone_number,website,opening_hours,reviews"
)

def place_details(pid):
    params = {"place_id": pid, "fields": DETAIL_FIELDS, "key": API_KEY}
    return SESSION.get(BASE_PLACE_DETAILS, params=params, timeout=(10, 30)).json()

def _details_with_backoff(pid, sleep_between=DETAILS_SLEEP):
    backoff = 1.0
    for _ in range(8):
        try:
            d = place_details(pid)
        except (requests.exceptions.SSLError,
                requests.exceptions.ConnectionError,
                requests.exceptions.ReadTimeout,
                requests.exceptions.ChunkedEncodingError):
            time.sleep(backoff); backoff = min(backoff*1.7, 20.0); continue
        status = d.get("status", "OK")
        if status == "OK": return d
        if status in ("OVER_QUERY_LIMIT", "UNKNOWN_ERROR"):
            time.sleep(backoff); backoff = min(backoff*1.7, 20.0); continue
        return d  
    return {"status":"SKIPPED"}

def enrich_places_with_details(df, limit=None, sleep_between=DETAILS_SLEEP):
    rows = df.to_dict(orient="records")
    if limit: rows = rows[:limit]
    enriched, reviews = [], []
    for p in rows:
        pid = p["place_id"]
        d = _details_with_backoff(pid, sleep_between=sleep_between)
        if d.get("status") != "OK":
            continue
        res = d.get("result") or {}
        loc = (res.get("geometry", {}) or {}).get("location", {}) or {}
        enriched.append({
            "place_id": pid,
            "name": res.get("name") or p.get("name"),
            "lat": loc.get("lat") or p.get("lat"),
            "lng": loc.get("lng") or p.get("lng"),
            "address": res.get("formatted_address") or p.get("address"),
            "google_types": ",".join(res.get("types", [])) or p.get("google_types"),
            "rating": res.get("rating"),
            "user_ratings_total": res.get("user_ratings_total"),
            "phone": res.get("international_phone_number"),
            "website": res.get("website"),
            "has_opening_hours": bool(res.get("opening_hours")),
            "source_category": p.get("source_category"),
        })
        for rv in (res.get("reviews") or []):
            text = (rv.get("text") or "").strip()
            tsec = rv.get("time")
            rid = hashlib.sha256(f"{pid}|{tsec}|{text[:64]}".encode("utf-8")).hexdigest()[:24]
            reviews.append({
                "review_id": rid,
                "place_id": pid,
                "author_name": rv.get("author_name"),
                "rating": rv.get("rating"),
                "text": text,
                "time": tsec,                  # unix seconds 
                "relative_time": rv.get("relative_time_description"),
                "language": rv.get("language"),
                
            })
        time.sleep(sleep_between)
    return pd.DataFrame(enriched), pd.DataFrame(reviews)

Pilot and Full enrichment (guarded)

In [37]:
# Optional pilot (kept for demonstration)
# df_det_pilot, df_rev_pilot = enrich_places_with_details(df_places, limit=100)
# print(df_det_pilot.shape, df_rev_pilot.shape)

if RUN_DETAILS:
    df_det_all, df_rev_all = enrich_places_with_details(df_places, limit=None)
    df_det_all.to_csv(PROC/"auckland_places_details.csv", index=False)
    df_rev_all.to_csv(PROC/"auckland_reviews.csv", index=False)
    print("Saved processed details & reviews.")
else:
    # ==== Load processed details & reviews safely ====
    det_p = PROC / "auckland_places_details.csv"
    rev_p = PROC / "auckland_reviews.csv"

    def safe_load_csv(path, name):
        """Try to load a CSV, but return empty DataFrame if missing/empty."""
        if path.exists() and os.path.getsize(path) > 0:
            try:
                df = pd.read_csv(path)
                print(f"Loaded {name}: {df.shape}")
                return df
            except Exception as e:
                print(f" Failed to read {name} ({path}): {e}")
                return pd.DataFrame()
        else:
            print(f" {name} not found or empty at {path}")
            return pd.DataFrame()

    df_det_all = safe_load_csv(det_p, "places_details")
    df_rev_all = safe_load_csv(rev_p, "reviews")

    print("Processed DataFrames summary:")
    print(" - places_details:", df_det_all.shape)
    print(" - reviews:", df_rev_all.shape)


Loaded places_details: (1528, 13)
Loaded reviews: (6306, 9)
Processed DataFrames summary:
 - places_details: (1528, 13)
 - reviews: (6306, 9)


Clean, map categories, basic QC (single source of truth)

In [29]:
# Places
places = (df_det_all
          .dropna(subset=["place_id"])
          .drop_duplicates(subset=["place_id"])
          .copy())

for col in ["lat","lng","rating"]:
    if col in places.columns:
        places[col] = pd.to_numeric(places[col], errors="coerce")
if "user_ratings_total" in places.columns:
    places["user_ratings_total"] = pd.to_numeric(places["user_ratings_total"], errors="coerce").fillna(0).astype(int)

places["category"] = places["types"].apply(map_to_category)

keep_places = ["place_id","name","lat","lng","address","category","types",
               "rating","user_ratings_total","phone","website","has_opening_hours","source_category"]
places = places[[c for c in keep_places if c in places.columns]]


Total places: 1528
Total reviews: 4688
Places with ≥1 review: 1108
Places missing lat/lng: 0

Places by category:
 category
park                  542
tourist attraction    341
restaurant            148
shopping mall         115
Name: place_id, dtype: int64

Top review languages:
 language
en       4631
en-US      55
pt          1
ko          1
Name: count, dtype: int64


In [ ]:
# ==== Normalize reviews safely (handles empty/missing columns) ====

reviews = df_rev_all.copy()

# If there are no rows at all, keep going without crashing
if reviews.empty:
    print("⚠️ reviews is empty (df_rev_all had no rows) — continuing")
else:
    # Helper to map/alias columns if original names differ
    def ensure_col(df, out_col, candidates, default=None):
        for c in candidates:
            if c in df.columns:
                df[out_col] = df[c]
                return
        df[out_col] = default

    # Map common variants -> canonical names used downstream
    ensure_col(reviews, "text", ["text", "review_text", "content", "body", "comment"], default="")
    ensure_col(reviews, "publish_time_utc", ["publish_time_utc", "time_utc", "time", "timestamp"], default=None)
    ensure_col(reviews, "rating", ["rating", "stars", "score"], default=None)
    ensure_col(reviews, "lang", ["lang", "language"], default="en")
    ensure_col(reviews, "source", ["source"], default="google_places")

    # Ensure we have a place_id column (most pipelines do)
    if "place_id" not in reviews.columns:
        print("⚠️ reviews has no place_id — downstream join to lat/lng/name may be limited")

    # Clean text and drop empty texts
    reviews["text"] = (
        reviews["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    )
    reviews = reviews[reviews["text"] != ""].reset_index(drop=True)

    # Review ID: create if missing or blank
    if "review_id" not in reviews.columns:
        reviews["review_id"] = reviews.apply(
            lambda r: make_review_id(r.get("place_id", ""), r.get("publish_time_utc", ""), r.get("text", "")),
            axis=1,
        )
    else:
        mask = reviews["review_id"].isna() | (reviews["review_id"].astype(str).str.strip() == "")
        if mask.any():
            reviews.loc[mask, "review_id"] = reviews[mask].apply(
                lambda r: make_review_id(r.get("place_id", ""), r.get("publish_time_utc", ""), r.get("text", "")),
                axis=1,
            )

    # Add alias for UI
    if "review_time" not in reviews.columns and "publish_time_utc" in reviews.columns:
        reviews["review_time"] = reviews["publish_time_utc"]

    # If lat/lng/name missing in reviews, try to enrich from places details
    need_latlng = ("lat" not in reviews.columns) or ("lng" not in reviews.columns)
    need_name = ("name" not in reviews.columns)
    if (need_latlng or need_name) and not df_det_all.empty and "place_id" in reviews.columns and "place_id" in df_det_all.columns:
        cols = [c for c in ["place_id", "name", "lat", "lng"] if c in df_det_all.columns]
        reviews = reviews.merge(
            df_det_all[cols].drop_duplicates(subset=["place_id"]),
            on="place_id",
            how="left"
        )

    # Rating to numeric (if present)
    if "rating" in reviews.columns:
        reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")

    # Final de-dupe
    if "review_id" in reviews.columns:
        reviews = reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

print("reviews shape:", reviews.shape)


In [ ]:
# Coverage stats
print("Total places:", places["place_id"].nunique())
print("Total reviews:", len(reviews))
print("Places with ≥1 review:", reviews["place_id"].nunique())
missing_coords = places[places["lat"].isna() | places["lng"].isna()]
print("Places missing lat/lng:", len(missing_coords))
print("\nPlaces by category:\n", places.groupby("category")["place_id"].nunique().sort_values(ascending=False))
if "language" in reviews.columns:
print("\nTop review languages:\n", reviews["language"].value_counts().head(10))

Final freeze 

In [31]:
places.to_csv(FINAL/"places.csv", index=False)
reviews.to_csv(FINAL/"reviews.csv", index=False)
print("Saved final snapshots →", FINAL/"places.csv", "and", FINAL/"reviews.csv")

Saved final snapshots → data\final\places.csv and data\final\reviews.csv


Hand-off file for NLP

In [ ]:
Final summary

In [33]:
print("Final places:", places["place_id"].nunique())
print("Final reviews:", len(reviews))
print("\nPlaces by category:")
print(places.groupby("category")["place_id"].nunique().sort_values(ascending=False))


Final places: 1528
Final reviews: 4688

Places by category:
category
park                  542
tourist attraction    341
restaurant            148
shopping mall         115
Name: place_id, dtype: int64


In [35]:
import os, json, hashlib
from datetime import datetime, timezone
import pandas as pd

os.makedirs("data/final", exist_ok=True)

# ---- Pick the most recent reviews DataFrame available ----
_reviews_candidates = ["reviews", "df_reviews_all", "df_reviews_pilot", "df_reviews"]
_reviews_df = None
for _name in _reviews_candidates:
    if _name in globals():
        _reviews_df = eval(_name).copy()
        break

# If nothing in memory, try to load an existing file to avoid failure
if _reviews_df is None:
    try:
        _reviews_df = pd.read_csv("data/final/reviews.csv")
    except Exception:
        raise RuntimeError(
            "No reviews DataFrame found in memory and data/final/reviews.csv does not exist yet."
        )

# ---- Ensure essential columns & IDs ----
if "review_id" not in _reviews_df.columns:
    def _mk_id(row):
        base = f"{row.get('place_id','')}|{row.get('publish_time_utc', row.get('time',''))}|{str(row.get('text',''))[:64]}"
        return hashlib.sha256(base.encode("utf-8")).hexdigest()[:24]
    _reviews_df["review_id"] = _reviews_df.apply(_mk_id, axis=1)

# Create review_time alias expected by some UIs
if "publish_time_utc" in _reviews_df.columns and "review_time" not in _reviews_df.columns:
    _reviews_df["review_time"] = _reviews_df["publish_time_utc"]

# Normalize dtypes a bit (optional)
if "rating" in _reviews_df.columns:
    _reviews_df["rating"] = pd.to_numeric(_reviews_df["rating"], errors="coerce")

# De-dupe by review_id
_reviews_df = _reviews_df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

# ---- Compute "new rows" vs existing to report items_upserted ----
existing_path = "data/final/reviews.csv"
existing_ids = set()
if os.path.exists(existing_path):
    try:
        _old = pd.read_csv(existing_path, usecols=["review_id"])
        existing_ids = set(_old["review_id"].astype(str))
    except Exception:
        pass
new_count = int((_reviews_df["review_id"].astype(str).isin(existing_ids) == False).sum())

# ---- Save reviews ----
_reviews_df.to_csv(existing_path, index=False)

# ---- Places: if you already wrote earlier, do nothing. If not, try to save safely. ----
# (Optional) if 'places' exists and you want to standardize:
# if "places" in globals():
#     places.to_csv("data/final/places.csv", index=False)

# ---- Update runs.json ----
run_info = {
    "last_successful_run_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "items_upserted": new_count,
    "notes": "refresh completed"
}
with open("data/final/runs.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved data/final/reviews.csv and updated data/final/runs.json")


✅ Saved data/final/reviews.csv and updated data/final/runs.json
